# Experimento D3 · ETL en Spark frente a ETL en SQL Server con más volumen
**Responsable:** Huaynate

## 1. Objetivo
Comparar cuánto tarda cargar y transformar el CSV (real + sintético) en Spark y en SQL Server cuando el volumen crece de 1x a 2x y 5x.

**Hipótesis:** Spark escala mejor que SQL Server al aumentar el volumen, aunque con poco volumen SQL Server puede ser igual o más rápido.

## 2. Configuración y condiciones de prueba
- Escenarios: 1x = CSV real; 2x = real + `sintetico_1M.csv`; 5x = real + `sintetico_4M.csv`.
- Spark: leer CSV → `limpiar_a_silver` → guardar tabla Delta.
- SQL Server: ETL de staging a Data Mart (dimensiones y hechos); el tiempo **no incluye** el `BULK INSERT` del CSV. Son los tiempos del experimento B1 (script `EXPERIMENTOS_BI.sql`).
- **Límites:** (1) el ETL de SQL construye dimensiones y hechos, el de Spark solo la tabla Silver; (2) el tiempo de Spark incluye leer el CSV y el de SQL no incluye cargarlo. Menciónalo en el análisis.

## 3. Procedimiento
1. Ejecuta `%run ./00_utilidades`.
2. Ejecuta las celdas de medición en orden. Cada medida hace **1 ejecución de calentamiento (no se cuenta) y 3 repeticiones**.
3. Los resultados se guardan solos en la tabla `resultados_experimentos`.
4. Completa el análisis (sección 5) con tus números.

**Evidencias a capturar:** la celda del ETL en Spark, la tabla de resultados y la captura de `log.EXPERIMENTOS` de SQL Server.

In [ ]:
%run ./00_utilidades

In [ ]:
EXP, RESP = "D3_etl_spark_vs_sql", "Huaynate"
ESCENARIOS = {
    "1x": [RUTA_CSV],
    "2x": [RUTA_CSV, f"{RUTA_EXPORT}/sintetico_1M.csv"],
    "5x": [RUTA_CSV, f"{RUTA_EXPORT}/sintetico_4M.csv"],
}

def etl_spark(rutas, destino):
    raw = None
    for r in rutas:
        d = spark.read.csv(r, header=True, inferSchema=False, sep=";", encoding="UTF-8").select(*COLUMNAS_ORIGINALES)
        raw = d if raw is None else raw.unionByName(d)
    guardar_tabla(limpiar_a_silver(raw), destino)

for esc, rutas in ESCENARIOS.items():
    destino = f"etl_tmp_{esc}"
    tiempos = medir(lambda r=rutas, d=destino: etl_spark(r, d))
    filas = leer_tabla(destino).count()
    f = registrar_resultado(EXP, f"spark_etl_{esc}", esc, filas, tiempos, "Spark", RESP, "CSV -> limpieza -> tabla Delta")
    print(f"Spark {esc}: {filas:,} filas | {f.media_s} s (± {f.desv_s}) | {f.throughput_filas_s:,} filas/s")

### Tiempos de SQL Server (los del experimento B1, ya cargados)
En SQL Server ejecuta: `SELECT variante, repeticion, segundos FROM log.EXPERIMENTOS WHERE experimento = 'B1' ORDER BY variante, repeticion;`
Si repites las mediciones en SQL, actualiza las cifras de la celda siguiente.

In [ ]:
# Tiempos medidos en SQL Server en el experimento B1 (log.EXPERIMENTOS), en segundos, y filas del staging cargado.
tiempos_sql = {
    "1x": ([44.675, 41.511, 40.761], 1_048_575),
    "2x": ([99.871, 95.372, 92.347], 2_048_575),
    "5x": ([248.525, 249.849, 239.417], 5_048_575),
}
for esc, (ts, filas) in tiempos_sql.items():
    if all(t == 0.0 for t in ts):
        print(f"SQL {esc}: pendiente, reemplaza los 0.0 por tus tiempos")
        continue
    registrar_resultado(EXP, f"sql_etl_{esc}", esc, filas, ts, "SQL Server", RESP, "ETL stg a dm (sin BULK INSERT)")

## 4. Resultados

In [ ]:
display(leer_tabla(T_RESULTADOS).where("experimento = 'D3_etl_spark_vs_sql'").orderBy("escala", "entorno"))

## 5. Comparación y análisis (completa con tus números)
- ¿En qué escala se cruzan las curvas (si se cruzan)?
- ¿Cuántas veces creció el tiempo de cada motor al pasar de 1x a 5x?
- Recuerda los límites: SQL construye dimensiones y hechos (sin contar la carga del CSV); Spark lee el CSV y construye solo la capa Silver.
- Conclusión: ¿cuándo conviene cada herramienta?